# 详细文档: Preissmann模型 - 河道断面

## 目的
此文档旨在详细介绍`preissmann_model`中河道断面（Cross-Section）的定义和使用。河道断面是构建一维水力模型的基础，它描述了河道在特定位置的几何形状，直接影响到水流的面积、湿周、水力半径等关键水力参数的计算。

此笔记本将展示如何：
1.  理解`BaseCrossSection`抽象基类。
2.  创建和使用`RectangularCrossSection`（矩形断面）。
3.  创建和使用`TrapezoidalCrossSection`（梯形断面）。
4.  可视化不同断面的几何形状及其随水深变化的性质。

## 1. 环境设置

首先，我们导入所需的库和我们自己的断面模块。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# 将项目根目录添加到Python路径
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))

from preissmann_model.cross_section import RectangularCrossSection, TrapezoidalCrossSection

## 2. 矩形断面 (RectangularCrossSection)

矩形断面是最简单的断面形式，仅由一个参数——河道宽度 `width` 定义。

In [ ]:
# 创建一个宽度为10米的矩形断面
rect_cs = RectangularCrossSection(width=10)

# 计算不同水深下的水力参数
water_depth = 2.0
area = rect_cs.area(water_depth)
top_width = rect_cs.top_width(water_depth)
wetted_perimeter = rect_cs.wetted_perimeter(water_depth)

print(f"矩形断面 (宽度=10m) 在水深为 {water_depth}m 时的参数:")
print(f"- 过水面积: {area:.2f} m^2")
print(f"- 水面宽度: {top_width:.2f} m")
print(f"- 湿周: {wetted_perimeter:.2f} m")

### 可视化矩形断面

In [ ]:
def plot_cs(ax, cs, max_y, title):
    y_values = np.linspace(0, max_y, 100)
    x_coords = [cs.top_width(y) / 2 for y in y_values]
    
    x_plot = np.concatenate([-np.array(x_coords), np.flip(x_coords)])
    y_plot = np.concatenate([y_values, np.flip(y_values)])
    
    ax.fill(x_plot, y_plot, 'lightblue', label='Wetted Area')
    ax.plot([-cs.top_width(0)/2, cs.top_width(0)/2], [0, 0], 'k-', linewidth=2, label='Bottom') # Bottom
    if isinstance(cs, RectangularCrossSection):
        ax.plot([-cs.width/2, -cs.width/2], [0, max_y], 'k-') # Left wall
        ax.plot([cs.width/2, cs.width/2], [0, max_y], 'k-') # Right wall
    elif isinstance(cs, TrapezoidalCrossSection):
        ax.plot([-cs.top_width(max_y)/2, -cs.bottom_width/2], [max_y, 0], 'k-') # Left wall
        ax.plot([cs.top_width(max_y)/2, cs.bottom_width/2], [max_y, 0], 'k-') # Right wall

    ax.set_title(title)
    ax.set_xlabel('Horizontal Distance (m)')
    ax.set_ylabel('Water Depth (m)')
    ax.set_aspect('equal', 'box')
    ax.grid(True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# 可视化矩形断面
plot_cs(ax1, rect_cs, max_y=3.0, title='Rectangular CS (Width=10m)')

## 3. 梯形断面 (TrapezoidalCrossSection)

梯形断面是更常见的河道形式，由两个参数定义：
- `bottom_width`: 河底宽度。
- `side_slope`: 边坡系数 `z`，表示水平方向`z`对应垂直方向`1`的坡度（zH:1V）。

In [ ]:
# 创建一个底宽为8米，边坡为2:1的梯形断面
trap_cs = TrapezoidalCrossSection(bottom_width=8, side_slope=2)

# 计算不同水深下的水力参数
water_depth = 2.0
area = trap_cs.area(water_depth)
top_width = trap_cs.top_width(water_depth)
wetted_perimeter = trap_cs.wetted_perimeter(water_depth)

print(f"梯形断面 (底宽=8m, 边坡=2) 在水深为 {water_depth}m 时的参数:")
print(f"- 过水面积: {area:.2f} m^2")
print(f"- 水面宽度: {top_width:.2f} m")
print(f"- 湿周: {wetted_perimeter:.2f} m")

### 可视化梯形断面

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
plot_cs(ax, trap_cs, max_y=3.0, title='Trapezoidal CS (BW=8m, Z=2)')

## 4. 水力参数随水深变化

我们可以绘制出关键水力参数（如过水面积、水力半径）随水深变化的曲线，以更好地理解不同断面形状的水力特性。

In [ ]:
depths = np.linspace(0.1, 5, 50)
rect_areas = [rect_cs.area(y) for y in depths]
trap_areas = [trap_cs.area(y) for y in depths]
rect_hr = [rect_cs.hydraulic_radius(y) for y in depths]
trap_hr = [trap_cs.hydraulic_radius(y) for y in depths]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# 绘制面积-水深关系
ax1.plot(depths, rect_areas, 'b-', label='Rectangular (W=10)')
ax1.plot(depths, trap_areas, 'r--', label='Trapezoidal (BW=8, Z=2)')
ax1.set_title('Area vs. Water Depth')
ax1.set_xlabel('Water Depth (m)')
ax1.set_ylabel('Wetted Area (m^2)')
ax1.legend()
ax1.grid(True)

# 绘制水力半径-水深关系
ax2.plot(depths, rect_hr, 'b-', label='Rectangular (W=10)')
ax2.plot(depths, trap_hr, 'r--', label='Trapezoidal (BW=8, Z=2)')
ax2.set_title('Hydraulic Radius vs. Water Depth')
ax2.set_xlabel('Water Depth (m)')
ax2.set_ylabel('Hydraulic Radius (m)')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()